# Fine-Tune Llama 3 for CYOM Nutrition Coach

This notebook uses **Unsloth** to fine-tune Llama 3 (8B) 2x faster and with 80% less memory. 
It is designed to run on a **free Google Colab instance (Tesla T4 GPU)**.

### Steps:
1. Install Dependencies
2. Load Pretrained Model (Llama 3 8B)
3. Load Your Dataset (`training_data.jsonl`)
4. Fine-Tune the Model
5. Export to GGUF (for Ollama)

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Must install Unsloth specifically for Colab's T4 GPU
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 1. Load the Pretrained Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # Llama-3 8B 4bit quantized
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 2. Add LoRA Adapters (This makes fine-tuning possible on free GPUs)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### 3. Load Data
Upload your `training_data.jsonl` file to the Colab 'Files' tab (sidebar) before running this.

In [ ]:
from datasets import load_dataset

# formatting_prompts_func specific to ChatML / Llama-3 format
from unsloth.chat_templates import get_chat_template_tokenizer
tokenizer = get_chat_template_tokenizer(tokenizer)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Ensure you uploaded training_data.jsonl to the root content directory
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this for better results (e.g. 100-200)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Export to GGUF (for Ollama)
To run this model on your local machine using Ollama, we export it to GGUF format.

In [ ]:
# Save to GGUF
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

# You can now download the .gguf file from the 'model' directory in Colab files!